# Beanthentic AI Training Notebook

**Capstone use:** Train the **GI document ensemble** (bagging + boosting soft vote), export accuracy metrics, and evaluate the MoP qualitative IPOPHL engine.

**Product focus:** The admin app uses **GI document analysis only** (MoP qualitative review + optional document ensemble advisory).

**Ensemble theory in this project**
- **Bagging:** Random Forest + Extra Trees (reduce variance)
- **Boosting:** Gradient Boosting (reduce bias)
- **Combination:** Soft voting averages class probabilities → Ready / Not Ready advisory score

**Run Jupyter from the project root** (folder containing `web.py`).

| Section | Trains / evaluates |
|---------|-------------------|
| **A** | Document ensemble (JSON → `gi_document_model.joblib`) |
| **B** | MoP qualitative review (what IPOPHL cards show today) |
| **C** | Deploy + improvement workflow |


# Paper results snapshot (auto-refreshed 2026-08-20T13:27:08)

| Metric | Value |
|--------|-------|
| Model | Soft-voting ensemble (Random Forest + Extra Trees + Gradient Boosting) |
| Training samples | **259** (Ready 125 / Not Ready 134) |
| Official MoP files | 7 |
| Hold-out test samples | **52** (Not Ready 27 / Ready 25) |
| Hold-out test accuracy | **98.08%** |
| Cross-validation | **97.68% ± 3.08%** |
| Not Ready precision / recall / F1 | **1.000 / 0.963 / 0.981** |
| Ready precision / recall / F1 | **0.962 / 1.000 / 0.980** |
| Weighted precision / recall / F1 | **0.982 / 0.981 / 0.981** |
| Confusion matrix | `[[26, 1], [0, 25]]` |
| Training date | 2026-08-20T13:27:08.731308 |
| Artifact | `machinelearning/gi_document_model.joblib` |
| Metrics JSON | `machinelearning/document_training_results.json` |
| Figure | `machinelearning/document_confusion_matrix.png` |

**Label mapping:** `0 = Not Ready`, `1 = Ready`.

**Note for the paper:** Dashboard Ready/Not Ready is driven by the MoP qualitative engine; the ensemble is the advisory ML layer evaluated above.

In [ ]:
# Cell 1 — Setup paths (run first)
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import (
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
    VotingClassifier,
)
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import GridSearchCV, cross_val_score, train_test_split

# Project root = two levels up from this notebook
NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "web.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent.parent / "web.py").exists():
    PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
else:
    PROJECT_ROOT = NOTEBOOK_DIR  # adjust manually if needed

ML_DIR = PROJECT_ROOT / "machinelearning"
sys.path.insert(0, str(ML_DIR))
sys.path.insert(0, str(PROJECT_ROOT))

from ensemble_learning import (
    ENSEMBLE_DESCRIPTION,
    build_gi_ensemble,
    describe_ensemble,
    ensemble_feature_importances,
    ensemble_param_grid,
)

print("Project root:", PROJECT_ROOT)
print("ML dir:", ML_DIR)
print("Ensemble:", ENSEMBLE_DESCRIPTION)

Project root: C:\Users\Eliaza Mae Malibiran\OneDrive - Polytechnic University of the Philippines\Desktop\071626 - BEANTHENTIC\Beanthentic
ML dir: C:\Users\Eliaza Mae Malibiran\OneDrive - Polytechnic University of the Philippines\Desktop\071626 - BEANTHENTIC\Beanthentic\machinelearning
Ensemble: Soft-voting ensemble: Random Forest (bagging) + Extra Trees (bagging) + Gradient Boosting (boosting). Class probabilities are averaged; the majority-ready probability becomes the advisory ML score.


## Section A — IPOPHL document ensemble

**Production training data:** all available official MoP files, generated task examples, hard negatives, and local training uploads.  
**Training artifacts:** `machinelearning/training_data/ipophl_official_mop_dataset.csv` and `machinelearning/training_data/gi_documents_raw.json`

The IPOPHL module uses the official MoP qualitative review as the primary judge, with the document ensemble as the advisory validation layer.

**Rebuild the complete dataset and retrain the production model:**
```bash
python scripts/build_document_training_data.py --train --target 259
```

## Section B — MoP qualitative engine (production IPOPHL cards)

This is the **authoritative** Ready / Not Ready logic on the dashboard (`gi_reference_basis.py`).  
Use this section to prove accuracy against your real MoP uploads or an expert-labeled test set.

In [4]:
from ai_engine import GIAnalyzer
from config.ipophl_store import OFFICIAL_IPOPHL_TASK_IDS, list_documents, resolve_file_path
from machinelearning.gi_reference_basis import evaluate_against_reference

analyzer = GIAnalyzer(str(ML_DIR), auto_train=False)

rows = []
for tid in OFFICIAL_IPOPHL_TASK_IDS:
    docs = list_documents(task_id=tid, limit=5)
    if not docs:
        rows.append({"task_id": tid, "file": "(none)", "status": "—", "word_count": 0, "missing": ""})
        continue
    rec = docs[0]
    path = resolve_file_path(rec["file_uuid"], filename_hint=rec.get("original_filename"))
    text = analyzer.extract_text_from_file(str(path)) if path else ""
    review = evaluate_against_reference(text, task_id=tid, term_matches=analyzer._term_matches)
    rows.append({
        "task_id": tid,
        "file": rec.get("original_filename") or (path.name if path else ""),
        "status": review["status"],
        "word_count": review["word_count"],
        "missing": ", ".join(review.get("missing_requirements") or [])[:80],
    })

mop_df = pd.DataFrame(rows)
display(mop_df)
print("Ready count:", (mop_df["status"] == "Ready").sum(), "/", len(OFFICIAL_IPOPHL_TASK_IDS))

# Authoritative official MoP labels from the rebuilt manifest
man_df = pd.DataFrame(manifest.get("files") or [])
display(man_df)
if not man_df.empty:
    print(
        "Official MoP qualitative Ready:",
        int((man_df["status"] == "Ready").sum()),
        "/",
        len(man_df),
    )


task_id,file,status,word_count,missing
phase1-introduction,(none),—,0,
phase1-history,(none),—,0,
phase1-physical-link,(none),—,0,
phase2-general,(none),—,0,
phase2-specific,(none),—,0,
phase2-production,(none),—,0,
phase3-control,(none),—,0,


Ready count: 0 / 7


sample_id,task_id,path,status,word_count
n1,phase1-introduction,reference\mop\PART 1 - Justification for the Request for Protection-20260724T153341Z-1-001\PART 1 - Justification for the Request for Protection\Introduction & Reputation.docx,Ready,1114
n2,phase1-history,reference\mop\PART 1 - Justification for the Request for Protection-20260724T153341Z-1-001\PART 1 - Justification for the Request for Protection\History of Kapeng Barako.docx,Ready,3921
n3,phase1-physical-link,reference\mop\PART 1 - Justification for the Request for Protection-20260724T153341Z-1-001\PART 1 - Justification for the Request for Protection\Physical link to the territory.docx,Ready,2403
n4,phase2-general,reference\mop\PART 2 - Technical Part-20260724T153344Z-1-001\PART 2 - Technical Part\TECHNICAL - General Description.docx,Ready,608
n5,phase2-specific,reference\mop\PART 2 - Technical Part-20260724T153344Z-1-001\PART 2 - Technical Part\TECHNICAL - Specific Description of the Production.docx,Not Ready,52
n6,phase2-production,reference\mop\PART 2 - Technical Part-20260724T153344Z-1-001\PART 2 - Technical Part\TECHNICAL - The Production Process.docx,Not Ready,2817
n7,phase3-control,reference\mop\CONTROL & TRACEABILITY & LABELLING.docx,Not Ready,73


Official MoP qualitative labels (n=7): Ready=4, Not Ready=3


### Expert validation (recommended for capstone)

Create `machinelearning/training_data/ipophl_expert_labels.csv`:

```csv
file_uuid,task_id,expected_status,reviewer
88b93a36-...,phase1-introduction,Ready,LGU Reviewer
```

Then compute agreement rate in a new cell:

```python
import csv
labels_path = ML_DIR / "training_data" / "ipophl_expert_labels.csv"
# compare expected_status vs system status → accuracy, confusion matrix
```

## Section C — Deploy models & improve the workflow

### Deploy to production (run in terminal after notebook metrics look good)

```bash
# Rebuild all available training data + retrain the complete document ensemble
python scripts/build_document_training_data.py --train --target 259

# Re-score stored uploads after model or MoP changes
python scripts/build_document_training_data.py --reanalyze

# Restart app
python web.py
```

### Improvement cycle (document in Chapter 3 / Methodology)

1. Update the official Part 1 / Part 2 / Control documents or their labels  
2. Rebuild all available training data and retrain the document ensemble  
3. Check notebook metrics and confusion matrix  
4. Reanalyze stored IPOPHL uploads  
5. Export JSON metrics and figures for the paper  

Full written guide: `docs/JUPYTER_ML_TRAINING_GUIDE.md`

## Full Results — All Available Training Data

The following cells load the latest saved training report and display the complete dataset summary, per-class metrics, confusion matrix, and cross-validation results.

In [3]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

current_results_path = Path.cwd() / "machinelearning" / "document_training_results.json"
if not current_results_path.exists():
    current_results_path = Path.cwd().parent.parent / "machinelearning" / "document_training_results.json"

full_results = json.loads(current_results_path.read_text(encoding="utf-8"))
full_report = pd.DataFrame(full_results["classification_report"]).T

print("TRAINING DATASET")
print(f"Total samples: {full_results['sample_count']}")
print(f"Ready: {full_results['ready_count']}")
print(f"Not Ready: {full_results['not_ready_count']}")
print(f"Training source: {full_results['training_source']}")
print(f"Training date: {full_results['training_date']}")
print(f"Ensemble: {full_results['ensemble_method']}")
print("Label mapping: 0 = Not Ready, 1 = Ready")

print("\nMODEL PERFORMANCE")
print(f"Holdout accuracy: {full_results['accuracy']:.4f} ({full_results['accuracy']:.2%})")
print(f"Cross-validation mean: {full_results['cv_mean']:.4f} ({full_results['cv_mean']:.2%})")
print(f"Cross-validation standard deviation: {full_results['cv_std']:.4f} ({full_results['cv_std']:.2%})")

print("\nCLASSIFICATION REPORT")
display(full_report.round(4))

print("CONFUSION MATRIX")
confusion = np.array(full_results["confusion_matrix"])
display(pd.DataFrame(
    confusion,
    index=["Actual Not Ready", "Actual Ready"],
    columns=["Predicted Not Ready", "Predicted Ready"],
))

print("\nBEST PARAMETERS")
display(pd.DataFrame([full_results["best_params"]]))

TRAINING DATASET
Total samples: 259
Ready: 125
Not Ready: 134
Training source: ipophl_official_mop_dataset.csv
Training date: 2026-08-20T13:27:08.731308
Ensemble: soft_voting_bagging_boosting
Label mapping: 0 = Not Ready, 1 = Ready

MODEL PERFORMANCE
Holdout accuracy: 0.9808 (98.08%)
Cross-validation mean: 0.9768 (97.68%)
Cross-validation standard deviation: 0.0308 (3.08%)

CLASSIFICATION REPORT


,precision,recall,f1-score,support
0,1.0000,0.9630,0.9811,27.0000
1,0.9615,1.0000,0.9804,25.0000
accuracy,0.9808,0.9808,0.9808,0.9808
macro avg,0.9808,0.9815,0.9808,52.0000
weighted avg,0.9815,0.9808,0.9808,52.0000


CONFUSION MATRIX


,Predicted Not Ready,Predicted Ready
Actual Not Ready,26,1
Actual Ready,0,25



BEST PARAMETERS


,rf__max_depth,rf__min_samples_leaf,rf__min_samples_split,rf__n_estimators
0,12,1,2,100
